In [ ]:
from IPython import get_ipython
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'  #'last', 'last_expr'

%autosave 120

In [2]:
from transformers import AutoTokenizer

!pip install tiktoken
!pip install smart_open

TOKENIZER_PATH = 'tokenizer/'
tokenizer = AutoTokenizer.from_pretrained(
            TOKENIZER_PATH, use_fast=True, trust_remote_code=True
        )

/home/lishengping/miniconda3/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


SPECIAL_TOKENS: ((151643, '<|endoftext|>'), (151644, '<|im_start|>'), (151645, '<|im_end|>'), (151646, '<|extra_0|>'), (151647, '<|extra_1|>'), (151648, '<|extra_2|>'), (151649, '<|extra_3|>'), (151650, '<|extra_4|>'), (151651, '<|extra_5|>'), (151652, '<|extra_6|>'), (151653, '<|extra_7|>'), (151654, '<|extra_8|>'), (151655, '<|extra_9|>'), (151656, '<|extra_10|>'), (151657, '<|extra_11|>'), (151658, '<|extra_12|>'), (151659, '<|extra_13|>'), (151660, '<|extra_14|>'), (151661, '<|extra_15|>'), (151662, '<|extra_16|>'), (151663, '<|extra_17|>'), (151664, '<|extra_18|>'), (151665, '<|extra_19|>'), (151666, '<|extra_20|>'), (151667, '<|extra_21|>'), (151668, '<|extra_22|>'), (151669, '<|extra_23|>'), (151670, '<|extra_24|>'), (151671, '<|extra_25|>'), (151672, '<|extra_26|>'), (151673, '<|extra_27|>'), (151674, '<|extra_28|>'), (151675, '<|extra_29|>'), (151676, '<|extra_30|>'), (151677, '<|extra_31|>'), (151678, '<|extra_32|>'), (151679, '<|extra_33|>'), (151680, '<|extra_34|>'), (15168

In [3]:
import os
import json
from collections import defaultdict

import jax
import orbax
import orbax.checkpoint
from smart_open import open
from flax.traverse_util import flatten_dict, unflatten_dict
from jax.sharding import PartitionSpec as PS
import numpy as np
import re
from paxml.main import get_experiment
from praxis import pax_fiddle
import typing
from praxis import base_hyperparams
from paxml import tasks_lib
import jax.numpy as jnp
import flax.linen as nn
from jax.sharding import Mesh
from functools import partial
from jax.experimental.pjit import pjit
import flax
from typing import Dict
from praxis import py_utils
import tensorflow as tf



read_dir = "gs://llm_base_models_us-central2/v5p_256/7B/PileDCSlimLlama7B4Kx4x256x1v5p/checkpoints"
# read_dir = "gs://llm_base_models/v3_8_gen/7B/PileDCSlimLlama7B4Kx4x256x1Mini/checkpoints"

step_prefix = "checkpoint"
step_format_fixed_length = 8
load_step = 300000

options = orbax.checkpoint.CheckpointManagerOptions(
    step_prefix=step_prefix, step_format_fixed_length=step_format_fixed_length
)
item = {
    "state": orbax.checkpoint.Checkpointer(orbax.checkpoint.PyTreeCheckpointHandler())
}
mngr = orbax.checkpoint.CheckpointManager(read_dir, item, options)

if load_step is None:
    load_step = mngr.latest_step()

checkpoint_name = f"{step_prefix}_" + str(load_step).zfill(step_format_fixed_length)

print(f"checkpoint_name: {checkpoint_name}")
metadata_path = os.path.join(read_dir, checkpoint_name, "metadata/metadata")
print(f"metadata_path: {metadata_path}")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

flat_metadata = flatten_dict(metadata["train_state_metadata"])
unpadded_global_shapes = defaultdict(dict)
for k, v in flat_metadata.items():
    param_key, shape_dtype = k[:-1], k[-1]
    if shape_dtype in ["unpadded_shape", "dtype"]:
        unpadded_global_shapes[param_key][shape_dtype] = v
    shape_dtype = unpadded_global_shapes[param_key]
    if len(shape_dtype) == 2:
        shape_dtype = jax.ShapeDtypeStruct(
            shape=shape_dtype["unpadded_shape"], dtype=shape_dtype["dtype"]
        )
        unpadded_global_shapes.update({param_key: shape_dtype})

# load model
unflat_unpadded_global_shapes = unflatten_dict(unpadded_global_shapes)
with jax.default_device(jax.devices("cpu")[0]):
    weights = mngr.restore(load_step, items={"state": unflat_unpadded_global_shapes})

2024-06-07 10:31:17.700714: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/usr/local/lib


checkpoint_name: checkpoint_00300000
metadata_path: gs://llm_base_models_us-central2/v5p_256/7B/PileDCSlimLlama7B4Kx4x256x1v5p/checkpoints/checkpoint_00300000/metadata/metadata


ERROR:absl:Checkpoint structure file does not exist at gs://llm_base_models_us-central2/v5p_256/7B/PileDCSlimLlama7B4Kx4x256x1v5p/checkpoints/checkpoint_00300000/state. Attempting to assume an implicit tree structure.
I0000 00:00:1717756324.767163  230379 gcs_resource.cc:99] Using default AdmissionQueue with limit 32
I0000 00:00:1717756324.770740  231383 google_auth_provider.cc:179] Running on GCE, using service account 887571727717-compute@developer.gserviceaccount.com
tcmalloc: large alloc 2491416576 bytes == 0x92308000 @  0x7facc5432680 0x7facc5452ff4 0x7fac9aac57a0 0x7fac9aac5832 0x7fac99f535b0 0x7fac9a70029c 0x7fac9a703091 0x7fac9a9d2344 0x7fac9b06c510 0x7facc53f8609 0x7facc51c1133
tcmalloc: large alloc 2491416576 bytes == 0x179d16000 @  0x7facc5432680 0x7facc5452ff4 0x7fac9aac57a0 0x7fac9aac5832 0x7fac99f535b0 0x7fac9a70029c 0x7fac9a703091 0x7fac9a9d2344 0x7fac9b06c510 0x7facc53f8609 0x7facc51c1133


In [9]:
partition_rules = (
            # embeddings
            ("lm/embedding_lookup/emb_var", PS("mdl", "data")),
            # atention
            ("self_attention/(query|key|value)/w", PS(None, "data", None, "mdl")),
            ("self_attention/post/w", PS(None, "mdl", "data", None)),
            ("self_attention/dyn_w_proj/dd", PS(None, "mdl", None, None)),
            ("self_attention/dyn_w_proj/dw1", PS(None, "mdl", None, None, None)),
            ("self_attention/dyn_w_proj/qkw", PS(None, None, None, None, None, None)),
    
            # mlp
            ("ffn_layer1/linear/w", PS(None, "data", "mdl")),
            ("ffn_layer1_gate/linear/w", PS(None, "mdl", "data")),
            ("ffn_layer2/linear/w", PS(None, "data", "mdl")),
            # layer norms
            ("layer_norm/scale", PS(None)),
            ("ff_layer/layer_norm", PS(None)),
            # output head
            ("lm/final_ln/scale", PS(None)),
            ("logits_ffn/linear", PS("data", "mdl")),
            ('.*', PS(None)),
        )

def tree_path_to_string(path, sep=None):
    keys = []
    for key in path:
        if isinstance(key, jax.tree_util.SequenceKey):
            keys.append(str(key.idx))
        elif isinstance(key, jax.tree_util.DictKey):
            keys.append(str(key.key))
        elif isinstance(key, jax.tree_util.GetAttrKey):
            keys.append(str(key.name))
        elif isinstance(key, jax.tree_util.FlattenedIndexKey):
            keys.append(str(key.key))
        else:
            keys.append(str(key))
    if sep is None:
        return tuple(keys)
    return sep.join(keys)


def named_tree_map(f, tree, *rest, is_leaf=None, sep=None):
    return jax.tree_util.tree_map_with_path(
        lambda path, x, *r: f(tree_path_to_string(path, sep=sep), x, *r),
        tree, *rest,
        is_leaf=is_leaf
    )


def match_partition_rules(rules, params):
    def get_partition_spec(name, leaf):
        if len(leaf.shape) == 0 or np.prod(leaf.shape) == 1:
            return PS()
        for rule, ps in rules:
            if re.search(rule, name) is not None:
                return ps
        raise ValueError(f'Partition rule not found for param: {name}')
    return named_tree_map(get_partition_spec, params, sep='/')

instantiate = base_hyperparams.instantiate

# exp = 'PileDCSlimLlama7B4Kx4x256x1Mini'
exp = 'PileDCSlimLlama7B4Kx4x256x1'
experiment_config = get_experiment(f'paxml.tasks.lm.params.c4.{exp}')()
experiment_config.ICI_MESH_SHAPE = [1, 8, 1]
experiment_config.PERCORE_BATCH_SIZE = 1
experiment_config.QUERY_CHUNK_SIZE = 256
experiment_config.USE_STATIC_W = True

task_p = experiment_config.task()
task_p = typing.cast(pax_fiddle.Config[tasks_lib.SingleTask], task_p)
# task_p.model.fprop_dtype = jnp.bfloat16 # jnp.dtype(task_p.model.fprop_dtype)
task_p.model.fprop_dtype = jnp.bfloat16 # jnp.dtype(task_p.model.fprop_dtype)

jax_task = instantiate(task_p)

params = weights['state']['mdl_vars']
params_specs = jax.eval_shape(lambda x: x, params)
train_state_partition = match_partition_rules(partition_rules, params_specs)
# state_logical_annotations = nn.get_partition_spec(weights_specs) # to PartitionSpec

dims = [1, 8, 1]
dim_names = ['replica', 'data', 'mdl']
mesh = Mesh(np.array(jax.devices()).reshape(dims), dim_names)
params_shard = jax.tree_map(lambda x: jax.sharding.NamedSharding(mesh, x), train_state_partition)
model = jax_task.model


In [10]:
@flax.struct.dataclass
class GreedyState:
    cur_len: jnp.ndarray
    sequences: jnp.ndarray
    running_token: jnp.ndarray
    is_sent_finished: jnp.ndarray
    params: Dict[str, jnp.ndarray]


@partial(
    pjit,
    in_shardings=(PS(), params_shard),
    out_shardings=(PS())
)
def generate(input_batch, params):
    input_ids = input_batch.ids
    pad_token_id = 0
    eos_token_id = 151643
    max_length = 50
    batch_size, cur_len = input_ids.shape
    eos_token_id = jnp.array(eos_token_id, dtype=jnp.int32 if eos_token_id is not None else None)
    pad_token_id = jnp.array(pad_token_id, dtype=jnp.int32)
    cur_len = jnp.array(cur_len)
    is_sent_finished = jnp.zeros((batch_size,), dtype=jnp.bool_)
    
    sequences = jnp.full((batch_size, max_length), pad_token_id, dtype=jnp.int32)
    sequences = jax.lax.dynamic_update_slice(sequences, input_ids, (0, 0))
    state = GreedyState(
        cur_len=cur_len,
        sequences=sequences,
        running_token=input_batch,
        is_sent_finished=is_sent_finished,
        params=params,
    )
    
    def greedy_search_cond_fn(state):
        has_reached_max_length = state.cur_len == max_length
        all_sequence_finished = jnp.all(state.is_sent_finished)
        # 所有句子遇到eos或者达到最大生成长度则结束
        finish_generation = jnp.logical_or(has_reached_max_length, all_sequence_finished)
        return ~finish_generation
        
    def greedy_search_body_fn(state):
        params = state.params
        outputs, cache = model.apply(params, state.running_token, method=model.__call__, mutable=['cache'])
        if 'cache' in cache:
            params['cache'] = cache['cache']
        print('outputs: ' ,outputs, len(outputs))
        print('cache: ' ,cache, len(cache))
        
        last_token_logit = outputs.probs[:, -1]
        next_token = jnp.argmax(last_token_logit, axis=-1)
        next_token = next_token * ~state.is_sent_finished + pad_token_id * state.is_sent_finished
        next_is_sent_finished = state.is_sent_finished | (next_token == eos_token_id)
        next_token = next_token[:, None]
        next_sequences = jax.lax.dynamic_update_slice(state.sequences, next_token, (0, state.cur_len))

        next_token = format_input(next_token, state.cur_len)
        next_token.segment_pos = state.cur_len.repeat(batch_size).reshape(-1, 1)
        
        return GreedyState(
            cur_len=state.cur_len + 1,
            sequences=next_sequences,
            running_token=next_token,
            is_sent_finished=next_is_sent_finished,
            params=params,
        )
    
    if input_ids.shape[1] > 1:
        state = greedy_search_body_fn(state)
    state = jax.lax.while_loop(greedy_search_cond_fn, greedy_search_body_fn, state)
    sequences=state.sequences
    return sequences


def format_input(input_ids, start):
    input_len = 1
    input_batch = NestedMap()
    input_batch.ids = input_ids
    input_batch.labels =None
    input_batch.weights = input_ids >= 0
    input_batch.paddings = jnp.zeros_like(input_batch.ids)
    input_batch.segment_ids = jnp.ones_like(input_batch.ids)
    # 确保 start 是具体值
    start = jax.lax.convert_element_type(start, jnp.int32)
    input_len = jax.lax.convert_element_type(input_len, jnp.int32)
    
    # pos = jnp.arange(start, start + input_len)
    # print(start, input_len)
    # input_batch.segment_pos = input_batch.segment_ids * pos
    return input_batch


def format_input0(input_ids, start):
   
    input_batch = NestedMap()
    input_ids = jnp.pad(input_ids, ([0, 0], [0, 1]))
    input_batch.labels =input_ids[:, 1:]
    input_ids = input_ids[:, :-1]
    input_len = input_ids.shape[1]
    input_batch.ids = input_ids
    input_batch.weights = input_ids >= 0
    input_batch.paddings = jnp.zeros_like(input_batch.ids)
    input_batch.segment_ids = jnp.ones_like(input_batch.ids)

    # 确保 start 是具体值
    start = jax.lax.convert_element_type(start, jnp.int32)
    input_len = jax.lax.convert_element_type(input_len, jnp.int32)
    
    pos = jnp.arange(start, start + input_len)
    
    print(start, input_len)
    input_batch.segment_pos = input_batch.segment_ids * pos
    return input_batch

In [20]:
# NestedMap = py_utils.NestedMap

# pngkey = jax.random.key(0)
# vocab = 152064
# batch_size = 1
# # max_length = 100
# # input_len = 256
# # input_ids = jax.random.randint(pngkey, minval=1, maxval=vocab, shape=[batch_size, input_len])
# inp = '<|extra_0|>周杰'
# input_ids = jnp.array(tokenizer.encode(inp)).reshape(batch_size, -1)
# input_batch = format_input0(input_ids, 0)
# with mesh:
#     output = generate(input_batch, params)
#     output = jax.device_get(output)

In [6]:
import pickle


p = 'summary_tensors_601.pkl'
ori_summary_tensors_601 = pickle.load(open(p, 'rb'))


for key, value in ori_summary_tensors_601.items():
    print(key)

learning/applied_grad_norm_aggregate_scalar
learning/clipped_grad_norm_aggregate_scalar
learning/grad_scale_aggregate_scalar
learning/is_valid_step_aggregate_scalar
learning/lr_aggregate_scalar
learning/raw_grad_norm_aggregate_scalar
learning/var_norm_aggregate_scalar
lm/[lsp]input_ids_scalar
lm/[lsp]inputs_scalar
lm/[lsp]logits_scalar
lm/num_unpadded_tokens_scalar
lm/transformer/repeat/sub/x_layers_0/ff_layer/expert_to_token_score_0_scalar
lm/transformer/repeat/sub/x_layers_0/ff_layer/token_to_expert_score_0_scalar
lm/transformer/repeat/sub/x_layers_0/self_attention/[lsp]atten_context_atten_mask_0_scalar
lm/transformer/repeat/sub/x_layers_0/self_attention/[lsp]atten_context_key_0_scalar
lm/transformer/repeat/sub/x_layers_0/self_attention/[lsp]atten_context_logits1_0_scalar
lm/transformer/repeat/sub/x_layers_0/self_attention/[lsp]atten_context_logits_0_scalar
lm/transformer/repeat/sub/x_layers_0/self_attention/[lsp]atten_context_probs2_0_scalar
lm/transformer/repeat/sub/x_layers_0/self

In [24]:
@partial(
    pjit,
    in_shardings=(PS(), params_shard),
    out_shardings=(PS())
)
def eval(input_batch, params):
    (weighted_scalars, per_example_output), updated_vars = model.apply(params, 
                                         input_batch,  # 模型的输入
                                         # method=model.__call__, # 模型forward函数，如果没有传，默认self.__call__
                                         rngs={'default': jax.random.key(0), 'params': jax.random.key(0)},
                                         mutable=True) 
    return weighted_scalars

In [25]:
# tpu eval
NestedMap = py_utils.NestedMap
input_ids = ori_summary_tensors_601['lm/[lsp]input_ids_scalar']
input_batch = format_input0(input_ids, 0)
with mesh:
    weighted_scalars = eval(input_batch, params)

0 256


In [26]:
weighted_scalars

{'aux_loss': (Array(0, dtype=bfloat16), Array(1, dtype=bfloat16)),
 'avg_xent': (Array(2.688847, dtype=float32), Array(2048., dtype=float32)),
 'batch_avg_acc': (Array(0.4560547, dtype=float32),
  Array(2048., dtype=float32)),
 'batch_avg_xent': (Array(2.688847, dtype=float32),
  Array(2048., dtype=float32)),
 'fraction_of_correct_next_step_preds': (Array(0.4560547, dtype=float32),
  Array(2048., dtype=float32)),
 'log_pplx': (Array(2.688847, dtype=float32), Array(2048., dtype=float32)),
 'num_predictions': (Array(2048., dtype=float32), Array(1., dtype=float32)),
 'total_loss': (Array(2.688847, dtype=float32), Array(2048., dtype=float32))}

In [9]:
# # cpu eval
# inp = '<|extra_0|>周杰伦'
# NestedMap = py_utils.NestedMap
# batch_size = 1
# input_ids = jnp.array(tokenizer.encode(inp)).reshape(batch_size, -1)
# input_ids = ori_summary_tensors_601['lm/[lsp]input_ids_scalar']
# input_batch = format_input0(input_ids, 0)
# with jax.default_device(jax.devices("cpu")[0]):
#     # outputs, intermediates = model.apply(params, input_batch, mutable=['intermediates'])
#      (weighted_scalars, per_example_output), updated_vars = model.apply(params, 
#                                          input_batch, 
#                                          method=model.__call__,
#                                          rngs={'params': jax.random.key(0)},
#                                          mutable=True) 

0 256


tcmalloc: large alloc 3205341184 bytes == 0x123ece4000 @  0x7fbd00738680 0x7fbd00759824 0x7fbd00759b8a 0x7fbce5914280 0x7fbcdefb0434 0x7fbcdefa6b67 0x7fbcdefab551 0x7fbce3970c51 0x7fbcdef35147 0x7fbcdef36159 0x7fbcded821c5 0x7fbcded82087 0x7fbcded81b6c 0x7fbcded81af4 0x7fbcded43fbf 0x4fd907 0x4f705b 0x5098bf 0x4f2856 0x4fdd4f 0x4f08a9 0x4f63ad 0x507bb0 0x5cf913 0x50a259 0x4f08a9 0x4fdd4f 0x50a108 0x4f08a9 0x4fdd4f 0x7fbcdedf250b


In [24]:
# indices = jnp.argmax(updated_vars['summaries']['lm']['[lsp]logits_scalar'][:, -1], axis=-1)